In [ ]:
#Get youngest tip date
suppressMessages(mcc_metadata <- read_csv(mcc_metadata_path))
mcc_metadata$sampleCollectionDate <- ymd(mcc_metadata$sampleCollectionDate)
mcc_youngest_date <- max(mcc_metadata$sampleCollectionDate, na.rm = TRUE)
mcc_youngest_decimal <- decimal_date(mcc_youngest_date)

In [ ]:
mcc_tree <- read.beast(mcc_tree_path)

mcc_tree_plot <- ggtree(mcc_tree, mrsd=mcc_youngest_date) + 
  		theme_tree2() + # This ggtree theme will add a forward in time x-axis
  		labs(caption="years") # Add caption to indicate that the x-axis is in years
  

suppressMessages( 
  mcc_tree_plot <- mcc_tree_plot +
    geom_tiplab(as_ylab=TRUE) + # Add tip labels as y-axis labels
    geom_range(range = 'height_0.95_HPD', center = "height_median", color = 'blue', alpha = 0.2, linewidth = 2) + # Add 95% HPD intervals for node heights
    geom_text2(aes(label = node), data=mcc_tree_plot$data %>% filter(!isTip), hjust = -0.2, size = 3) # Add node labels for internal nodes
)

data_rescaled <- mcc_tree_plot$data |>
  filter(!isTip) |>
  mutate(height_median_rescaled = mcc_youngest_decimal - height_median) # Rescale median node heights to be in the same units as the x-axis (years before the youngest tip)
mcc_tree_plot <- mcc_tree_plot +
  geom_point(data = data_rescaled, aes(x = height_median_rescaled, y = y), color = "red", size = 2
  )  # Add points for median node heights

mcc_dotted_line_lst <- get_tip_data_and_xend(mcc_tree_plot) # Get data for plotting dotted lines from tips to x-axis
mcc_tree_plot <- mcc_tree_plot +
  geom_segment(
    data = mcc_dotted_line_lst$tip_data,
    aes(x = x, y = y, xend = mcc_dotted_line_lst$xend, yend = y),
    linetype = "dotted",
    color = "grey60",
    alpha = 0.5
  ) # Add dotted lines from tips to x-axis

mcc_tree_plot # Display the tree plot

In [ ]:
# Create a dataframe with node heights and branch lengths, including 95% HPD intervals and ranges for both heights and lengths, and convert heights to dates.
mcc_data <- mcc_tree_plot$data |>
  unnest_wider(
    c(
    length_0.95_HPD, length_range,
    height_0.95_HPD, height_range
  ), names_sep = "_") |> # Unnest the nested columns for HPD intervals and ranges, separating names with an underscore
  rename_with(~ sub("_1$", "_upper", .x), ends_with("_1")) |> # Rename columns ending with "_1" to end with "_lower"
  rename_with(~ sub("_2$", "_lower", .x), ends_with("_2")) |> # Rename columns ending with "_2" to end with "_upper"
  dplyr::select(
    node, parent, branch.length, label, height,
    height_0.95_HPD_lower, height_0.95_HPD_upper, height_median,
    height_range_lower, height_range_upper, length,
    length_0.95_HPD_lower, length_0.95_HPD_upper, length_median,
    length_range_lower, length_range_upper, isTip
  ) |> # Select and reorder columns to include node information, heights, lengths, and their respective HPD intervals and ranges
  mutate(across(
    starts_with("height"),
    ~ date_decimal(mcc_youngest_decimal - .x)
  )) |> # Convert height values to dates by subtracting the height from the youngest decimal date and converting to a date format
  mutate(across(
    starts_with("height"),
    ~ as_date(.x)
  )) # Ensure that the converted height values are in date to day and not including time format


write.csv(mcc_data, mcc_tree_data_path, row.names = FALSE)


Summary Stats Table of Node Estimates

In [ ]:
node_mcc_data <- mcc_data |>
  filter(!isTip) |>
  select(node, starts_with("height")) # Create table of only internal nodes and height-related columns

# Create HTML table with scrollable container for internal node heights and CIs
# Define columns needing wider width
wide_cols <- c("height_0.95_HPD_lower", "height_0.95_HPD_upper", "height_median",
               "height_range_lower", "height_range_upper")

# Create HTML table with custom column widths
scrollable_table <- tags$div(
  style = "height:300px; overflow-y:auto; border:1px solid #ccc; width:80%;",
  tags$table(
    style = "border-collapse: collapse; width: 100%;",
    tags$thead(
      tags$tr(
        lapply(names(node_mcc_data), function(col) {
          col_width <- if (col %in% wide_cols) "180px" else "90px"
          tags$th(style = paste0("border: 1px solid #ccc; padding: 4px; background: #f2f2f2; width:", col_width, ";"), col)
        })
      )
    ),
    tags$tbody(
      lapply(1:nrow(node_mcc_data), function(i) {
        tags$tr(
          mapply(function(val, col) {
            col_width <- if (col %in% wide_cols) "180px" else "90px"
            tags$td(style = paste0("border: 1px solid #ccc; padding: 4px; width:", col_width, ";"), val)
          }, node_mcc_data[i, ], names(node_mcc_data), SIMPLIFY = FALSE)
        )
      })
    )
  )
)

browsable(scrollable_table)